In [1]:
import random
import torch
from pathlib import PurePath

from code.utility.utils import project_root_path
from code.config import cfg, update_cfg
from code.data_loader import MyDataLoader
import code.baseline.cv.conse as conse
import code.baseline.cv.vgse as vgse
import code.baseline.cv.costa as costa
import code.utility.utils as utils
from code.utility import joint_latent
from code.utility.classifier import BASECLASSIFIER

In [2]:
cfg.dataset = "CUB" 
cfg.method = "icis" 
cfg.cv.image_embedding = "res101_finetuned" 
cfg.cv.class_embedding = "att" 
cfg.cos_sim_loss = True 
cfg.include_unseen = True 
cfg.num_layers = 2 
cfg.beta1 = 0.9 
cfg.lr = 0.00001 
cfg.batch_size = 16 
cfg.embed_dim = 2048 
cfg.strict_eval = True 
cfg.early_stopping_slope = True 
cfg.cv.calc_entropy = True

# Update from task-specific information
# Add task type: cv, nlp, graph
if cfg.dataset in ["AWA2", "CUB", "SUN"]:
    cfg.task_type = "cv"
else:
    raise ValueError("Unrecognized dataset {}".format(cfg.dataset))
# Path to datasets folder
cfg.data_root = str(PurePath(
    project_root_path,
    "data", cfg.task_type,
))
# Path to input folder
cfg.input_root = str(PurePath(
    project_root_path,
    "input", cfg.task_type,
))
# Path to save model checkpoints and results
cfg.out_root = str(PurePath(
    project_root_path,
    "output", cfg.task_type,
))

In [3]:
seed_list = [cfg.seed] if cfg.num_runs == 1 \
    else [random.randint(1, 10000) for _ in range(cfg.num_runs)]

accs_unseen_only, accs_gzsl, accs_unseen, accs_seen, hs = [], [], [], [], []
accs_unseen_only_std, accs_gzsl_std, accs_unseen_std, accs_seen_std, hs_std = [], [], [], [], []

acc_gzsl_seeds_avg, acc_seen_seeds_avg, acc_unseen_seeds_avg, H_seeds_avg, unseen_zsl_seeds_avg =\
    [], [], [], [], []

seed = seed_list[0]
utils.set_seed(seed=seed)

# load data
data = MyDataLoader(cfg)
num_class = len(data.seen_classes) + len(data.unseen_classes)

Loaded attributes / word embeddings with shape torch.Size([200, 312])


In [4]:
data.train_label.shape

torch.Size([7057])

In [5]:
data.train_label.unique().shape

torch.Size([150])

In [6]:
data.train_feature.shape

torch.Size([7057, 2048])

In [7]:
data.test_seen_feature.shape

torch.Size([1764, 2048])

In [8]:
data.test_unseen_feature.shape

torch.Size([2967, 2048])

In [9]:
data.seen_classes

tensor([  0,   1,   2,   3,   4,   5,   7,   8,   9,  10,  11,  12,  13,  14,
         15,  16,  17,  19,  21,  22,  23,  24,  25,  26,  27,  29,  30,  31,
         32,  34,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,
         48,  50,  51,  52,  53,  54,  56,  57,  58,  59,  60,  62,  63,  64,
         65,  66,  69,  70,  72,  73,  74,  75,  76,  77,  80,  81,  82,  83,
         84,  85,  88,  89,  91,  92,  93,  95,  96,  98, 100, 101, 102, 104,
        105, 106, 108, 109, 110, 111, 112, 113, 114, 116, 117, 118, 120, 122,
        125, 126, 127, 129, 130, 131, 132, 133, 134, 135, 136, 137, 139, 142,
        143, 144, 145, 146, 147, 148, 150, 152, 153, 154, 155, 157, 160, 161,
        162, 163, 164, 167, 168, 169, 171, 172, 174, 176, 177, 179, 180, 182,
        183, 185, 187, 189, 193, 195, 196, 197, 198, 199])

In [10]:
data.seen_classes.shape

torch.Size([150])

In [11]:
data.num_class

200

In [1]:
data.attribute.shape

NameError: name 'data' is not defined